# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [17]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)
ndf["Time"] = pd.to_datetime(ndf["Time"])

## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

### Total Number of Flows

Count the total number of flows in this trace.

In [18]:
# groupby source and destination ip address
flows = ndf.groupby(['Source', 'Destination'])
len(flows)

77

### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [19]:
flows['Length'].sum().sort_values(ascending=False)
# large flows look like they are from the same source and destination ip address, it's netflix sending the video data.

Source                                              Destination                            
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                              120607242
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                7138148
192.168.43.72                                       ipv4-c071-cdg001-ix.1.oca.nflxvideo.net      3357228
a23-57-80-120.deploy.static.akamaitechnologies.com  192.168.43.72                                1332086
ipv4-c063-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                 431178
                                                                                                 ...    
17.188.166.20                                       192.168.43.72                                    218
0.0.0.0                                             all-systems.mcast.net                            184
Netgear_bb:19:ee                                    Apple_01:4c:54  

### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [20]:
flows.size().sort_values(ascending=False)
# they are similar to the largest flows by bytes, but the distribution is a little different. somewhat similar ordinally but differnet orders of magnitude than the byte flow data 

Source                                              Destination                            
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                              80084
192.168.43.72                                       ipv4-c071-cdg001-ix.1.oca.nflxvideo.net    47902
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                               4873
192.168.43.72                                       ipv4-c069-cdg001-ix.1.oca.nflxvideo.net     3170
a23-57-80-120.deploy.static.akamaitechnologies.com  192.168.43.72                               1005
                                                                                               ...  
17.252.44.15                                        192.168.43.72                                  3
17.188.166.20                                       192.168.43.72                                  2
192.168.43.72                                       192.168.43.255                                 1

### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [22]:
### Duration

# Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

# What are the longest flows in the trace?

durations = (
    flows["Time"].agg(["min", "max"])
)

durations["duration"] = durations["max"] - durations["min"]

# Longest flows
durations.sort_values("duration", ascending=False).head(20)

,,min,max,duration
Source,Destination,,,
192.168.43.72,par10s38-in-f3.1e100.net,2018-02-11 08:10:00.861944,2018-02-11 08:18:16.313632,0 days 00:08:15.451688
par10s38-in-f3.1e100.net,192.168.43.72,2018-02-11 08:10:01.209115,2018-02-11 08:18:16.313540,0 days 00:08:15.104425
ns-vip-pro.paris.inria.fr,192.168.43.72,2018-02-11 08:10:00.853950,2018-02-11 08:18:13.599825,0 days 00:08:12.745875
192.168.43.72,ns-vip-pro.paris.inria.fr,2018-02-11 08:10:00.534682,2018-02-11 08:18:13.222075,0 days 00:08:12.687393
192.168.43.97,224.0.0.251,2018-02-11 08:10:03.514149,2018-02-11 08:18:10.219615,0 days 00:08:06.705466
192.168.43.72,224.0.0.251,2018-02-11 08:10:03.808095,2018-02-11 08:18:10.208890,0 days 00:08:06.400795
fe80::e6ce:8fff:fe01:4c54,ff02::fb,2018-02-11 08:10:03.808330,2018-02-11 08:18:10.209063,0 days 00:08:06.400733
192.168.43.72,ec2-34-252-77-54.eu-west-1.compute.amazonaws.com,2018-02-11 08:10:12.323784,2018-02-11 08:18:12.876528,0 days 00:08:00.552744
ec2-34-252-77-54.eu-west-1.compute.amazonaws.com,192.168.43.72,2018-02-11 08:10:12.468557,2018-02-11 08:18:12.876416,0 days 00:08:00.407859


## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

In [36]:
durations2 = pd.DataFrame(durations["duration"])
lenghts = pd.DataFrame(flows['Length'].sum())
pd.merge(durations2, lenghts, left_on="Source", right_on="Source")

,duration,Length
Source,,
0.0.0.0,0 days 00:07:27.081009,3032
0.0.0.0,0 days 00:07:27.081009,184
0.0.0.0,0 days 00:06:16.254753,3032
0.0.0.0,0 days 00:06:16.254753,184
104.31.113.215,0 days 00:00:02.685329,934
...,...,...
par10s29-in-f3.1e100.net,0 days 00:00:03.662220,1091
par10s29-in-f4.1e100.net,0 days 00:04:08.460231,71268
par10s38-in-f13.1e100.net,0 days 00:04:02.987165,7683


## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?